In [2]:
%load_ext autoreload
%autoreload 2

from seek_code import SEEK
from backbone import Backbone
from retrieval import Retrieval
from projector import Projector
from data_handler import EleHandler
from concept_head_tunneled import ConceptHeadTunneled, categorical_CE_loss, CrossedHeadNN, MultiHeadNN

from pathlib import Path
from torch.utils.data import DataLoader, Subset
import os,json
import pandas as pd


/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# The following function can create a new data dictionary with the whatever aggregation method you want

In [15]:
aggregate_from_vote_to_SEEK = SEEK.aggregate_from_vote_to_SEEK


def test_compute_traditional_seek_codes_basic():
    traditional_rules = {
        'sex': {'cutoff': 0.8, 'fraction': 2},
        'age': {'cutoff': 0.8, 'fraction': 1},
        'tusks': {'cutoff': 0.8, 'fraction': 1},
        'ear_most_prominent': {'cutoff': 0.9, 'fraction': 2},
        'ear_least_prominent': {'cutoff': 0.95, 'fraction': 2.5},
        'extremes': {'cutoff': 0.8, 'fraction': 1.5}
    }


    merged_df_mv, image_seek_df_mv = aggregate_from_vote_to_SEEK(
        rules=traditional_rules,
        export_path='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_traditional_rules.csv',
        drop_missing_ears=False
    )


    count, dist = 0, 0
    for subject_id, row in merged_df_mv.iterrows():
        old_code = SEEK(row['old-subject-SEEK'])
        new_code = SEEK(row['subject-SEEK'])
        x = SEEK.distance_3(old_code, new_code)
        dist += x
        count += 1


    print('The average distance from the old-school way to our new function is ', dist / count )


    merged_df_mv, image_seek_df_mv = aggregate_from_vote_to_SEEK(
        rules=traditional_rules,
        export_path='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_traditional_rules_without_ears.csv',
        drop_missing_ears=True
    )


    count, dist = 0, 0
    for subject_id, row in merged_df_mv.iterrows():
        old_code = SEEK(row['old-subject-SEEK'])
        new_code = SEEK(row['subject-SEEK'])
        x = SEEK.distance_3(old_code, new_code)
        dist += x
        count += 1


    print('If we detect the ears, the distance goes up slightly ', dist / count )


    majority_vote_rules = {
        'sex': {'cutoff': 0.51, 'fraction': 1},
        'age': {'cutoff': 0.51, 'fraction': 1},
        'tusks': {'cutoff': 0.51, 'fraction': 1},
        'ear_most_prominent': {'cutoff': 0.51, 'fraction': 1},
        'ear_least_prominent': {'cutoff': 0.51, 'fraction': 1},
        'extremes': {'cutoff': 0.51, 'fraction': 1}
    }


    merged_df_mv, image_seek_df_mv = aggregate_from_vote_to_SEEK(
        rules=majority_vote_rules,
        export_path='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_traditional_rules.csv',
        drop_missing_ears=True
    )


    count, dist = 0, 0
    for subject_id, row in merged_df_mv.iterrows():
        old_code = SEEK(row['old-subject-SEEK'])
        new_code = SEEK(row['subject-SEEK'])
        x = SEEK.distance_3(old_code, new_code)
        dist += x
        count += 1
        #print('old SEEK:', old_code, ' new SEEK:', new_code,' distance is' , x)


    print('And if we switch to majority rule, the distance is pretty different : ', dist / count )


test_compute_traditional_seek_codes_basic()



The average distance from the old-school way to our new function is  0.0
If we detect the ears, the distance goes up slightly  0.07579346281383272
If we detect the ears, the distance goes up slightly  0.07579346281383272
And if we switch to majority rule, the distance is pretty different :  0.5529843675983038
And if we switch to majority rule, the distance is pretty different :  0.5529843675983038


In [ ]:
def retrieval_vs_rule(rules):
    _, _ = aggregate_from_vote_to_SEEK(
    rules=traditional_rules,
    export_path='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_temp.csv',
    drop_missing_ears=True
    )


    dataset = EleHandler(subset='IDI_6', dictonary_path='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_temp.csv')
    train_indices, test_indices = dataset.split_perpendicular_to_elephants_and_encounters(split_sizes=[0.5,0.5], hour_delta = 0.2)

    train_subset = Subset(dataset, train_indices)
    test_subset = Subset(dataset, test_indices)

    train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_subset, batch_size=64, shuffle=True)

    code ='aggregation-test0'
    r = Retrieval()

    MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
    concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False, pretraining='concept_w')

    query_embeddings, query_labels = concept_head.collect_embeddings(test_loader, backbone=MD_for_concepts, intervention_fn=SEEK.perfect_correction, aggregate_seeks=False)
    gallery_embeddings, gallery_labels = concept_head.collect_embeddings(train_loader, backbone=MD_for_concepts, intervention_fn=SEEK.perfect_correction, aggregate_seeks=True)
    test_similarity_matrix = r.similarity_matrix(query_embeddings, gallery_embeddings, distance='seek_homemade_3')

    # Compute and print recalls
    test_recalls = {k: r.compute_recall_at_k(similarity_matrix=test_similarity_matrix, query_labels=query_labels, gallery_labels=gallery_labels, k=k) for k in [1,5,10,20,100]}

    return test_recalls


In [18]:
def distant_from_agg_to_orcale(rules):
    merged_df_mv,image_seek_df_mv = aggregate_from_vote_to_SEEK(
    rules=rules,
    export_path='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_temp.csv',
    drop_missing_ears=True
    )

    count, dist = 0, 0
    for subject_id, row in merged_df_mv.iterrows():
        old_code = SEEK(row['subject-SEEK'])
        new_code = SEEK(row['ele-SEEK'])
        x = SEEK.distance_3(old_code, new_code) 
        dist += x
        count +=1 

    return dist / count




avg_dist = distant_from_agg_to_orcale(traditional_rules)
print('The average distance from the oracle to our traditional aggregation function is ', avg_dist)

The average distance from the oracle to our traditional aggregation function is  1.070375414495517


In [ ]:
import math
import optuna
from optuna.samplers import TPESampler

# ---- Your objective (already provided) is assumed to be in scope:
# def distant_from_agg_to_orcale(rules): ...
# and aggregate_from_vote_to_SEEK, SEEK etc. are importable.

# Sensible default ranges per field (feel free to tweak)
_DEFAULT_BOUNDS = {
    "sex":                {"cutoff": (0.2, 0.99),  "fraction": (0.5, 3.5)},
    "age":                {"cutoff": (0.2, 0.99),  "fraction": (0.5, 3.5)},
    "tusks":              {"cutoff": (0.2, 0.99),  "fraction": (0.5, 3.5)},
    "ear_most_prominent": {"cutoff": (0.2, 0.995), "fraction": (0.5, 3.5)},
    "ear_least_prominent":{"cutoff": (0.1, 0.999), "fraction": (0.5, 3.5)},
    "extremes":           {"cutoff": (0.2, 0.99),  "fraction": (0.5, 3.5)},
}

def tune_rules(
    base_rules=  traditional_rules,
    *,
    bounds=None,
    n_trials: int = 150,
    seed: int = 0,
    storage: str | None = None,   # e.g., "sqlite:///tuning.db" to persist
    study_name: str | None = None
):
    """
    Minimze distant_from_agg_to_orcale(rules) over (cutoff, fraction) for each field.

    Args
    ----
    base_rules: dict | None
        Optional starting rules; fields not in bounds won't be tuned.
        If None, uses your current defaults (from the prompt).
    bounds: dict | None
        Optional per-field bounds overriding _DEFAULT_BOUNDS.
        Example:
          {
            "sex": {"cutoff": (0.6, 0.95), "fraction": (1.0, 3.0)},
            "ear_least_prominent": {"cutoff": (0.9, 0.999), "fraction": (1.0, 4.0)},
          }
    n_trials: int
        Number of trials to run.
    seed: int
        Seed for reproducibility.
    storage: str | None
        Optuna storage URI to persist results (e.g., SQLite).
    study_name: str | None
        Name of the study when using persistent storage.

    Returns
    -------
    best_rules: dict
        The tuned rules dict.
    best_value: float
        The best objective value found.
    study: optuna.study.Study
        The Optuna study object (for plots, importance, etc.).
    """
        

    tune_bounds = _DEFAULT_BOUNDS.copy()
    if bounds:
        # shallow merge: only override provided fields
        for k, v in bounds.items():
            tune_bounds[k] = {**tune_bounds.get(k, {}), **v}

    fields_to_tune = list(tune_bounds.keys())

    def objective(trial: optuna.trial.Trial) -> float:
        # Start from base_rules; replace tuned fields with suggestions
        rules = {k: dict(v) for k, v in base_rules.items()}

        for field in fields_to_tune:
            # Skip fields not present in base_rules (safety)
            if field not in rules:
                rules[field] = {}

            lo_c, hi_c = tune_bounds[field]["cutoff"]
            lo_f, hi_f = tune_bounds[field]["fraction"]

            # Cutoffs between 0 and 1 — search linearly near top end
            rules[field]["cutoff"] = trial.suggest_float(f"{field}.cutoff", lo_c, hi_c)

            # Fractions are positive, can vary a lot → log scale helps
            rules[field]["fraction"] = trial.suggest_float(
                f"{field}.fraction", lo_f, hi_f, log=True
            )

        # Evaluate your pipeline
        try:
            val = distant_from_agg_to_orcale(rules)
        except Exception as e:
            # If something goes wrong in data/aggregation, nuke this trial
            trial.set_user_attr("error", str(e))
            return -math.inf  # we're maximizing

        if val is None or (isinstance(val, float) and (math.isnan(val) or math.isinf(val))):
            return -math.inf

        return float(val)

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=seed, n_startup_trials=min(20, max(5, n_trials // 10))),
        storage=storage,
        study_name=study_name,
        load_if_exists=bool(storage and study_name),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    # Build the best rules dict back from base + best params
    best = base_rules.copy()
    for field in fields_to_tune:
        if f"{field}.cutoff" in study.best_params:
            best[field] = {
                "cutoff": study.best_params[f"{field}.cutoff"],
                "fraction": study.best_params[f"{field}.fraction"],
            }

    return best, study.best_value, study

best_rules, best_val, study = tune_rules(n_trials=500, seed=42)

print("Best value:", best_val)
print("Best rules:")
for k, v in best_rules.items():
    print(k, v)


from optuna.importance import get_param_importances
imps = get_param_importances(study)
print("\nParameter importances:")
for k, v in imps.items():
    print(f"{k}: {v:.3f}")


retrieval_vs_rule(best_rules)

[I 2025-10-24 11:49:38,514] A new study created in memory with name: no-name-dfb5d08b-781e-4bd7-84ff-6e74d8e81c9f
[I 2025-10-24 11:49:39,594] Trial 0 finished with value: 1.0175509237328506 and parameters: {'sex.cutoff': 0.4958866938894164, 'sex.fraction': 3.1799244451886666, 'age.cutoff': 0.7782752140310101, 'age.fraction': 1.6028582466136447, 'tusks.cutoff': 0.32325472594952487, 'tusks.fraction': 0.6773305553993897, 'ear_most_prominent.cutoff': 0.2461764716737186, 'ear_most_prominent.fraction': 2.697576486600797, 'ear_least_prominent.cutoff': 0.6404023955571447, 'ear_least_prominent.fraction': 1.9831736892630007, 'extremes.cutoff': 0.21626175049368396, 'extremes.fraction': 3.3009497973205284}. Best is trial 0 with value: 1.0175509237328506.
[I 2025-10-24 11:49:40,658] Trial 1 finished with value: 1.2386309805779379 and parameters: {'sex.cutoff': 0.8576296862323332, 'sex.fraction': 0.7558182437334618, 'age.cutoff': 0.3436417240936095, 'age.fraction': 0.7144384114388685, 'tusks.cutoff'

Best value: 0.8964116532449272
Best rules:
sex {'cutoff': 0.41697801059374917, 'fraction': 3.012295575491241}
age {'cutoff': 0.5786023905440678, 'fraction': 2.54969216141909}
tusks {'cutoff': 0.7576885534651434, 'fraction': 1.0888076241284204}
ear_most_prominent {'cutoff': 0.2000159893228821, 'fraction': 1.8661342333113067}
ear_least_prominent {'cutoff': 0.17997545975283208, 'fraction': 1.2312084593635206}
extremes {'cutoff': 0.9561782000781985, 'fraction': 0.7644556127889781}

Parameter importances:
sex.fraction: 0.474
age.fraction: 0.259
ear_most_prominent.fraction: 0.058
ear_most_prominent.cutoff: 0.054
ear_least_prominent.fraction: 0.040
sex.cutoff: 0.039
tusks.cutoff: 0.028
ear_least_prominent.cutoff: 0.021
extremes.cutoff: 0.016
age.cutoff: 0.007
tusks.fraction: 0.002
extremes.fraction: 0.001


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Concept head loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt


{1: 0.041888804265041886,
 5: 0.07616146230007616,
 10: 0.09748667174409749,
 20: 0.13175932977913177,
 100: 0.3153084539223153}

### RESULTS : I ran the search for 10'000 iterations and obtained the following rules

In [19]:
#Best value: 0.8883112269066938

optimized_voting_rules = {
'sex': {'cutoff': 0.7238117020284947, 'fraction': 3.2846153859098934},
'age': {'cutoff': 0.6080374539126856, 'fraction': 3.030614746927617},
'tusks': {'cutoff': 0.7864963346562002, 'fraction': 1.3312930166044628},
'ear_most_prominent':{'cutoff': 0.2133936013358265, 'fraction': 1.788490949619698},
'ear_least_prominent': {'cutoff': 0.1001741505264265, 'fraction': 1.6865733430829595},
'extremes': {'cutoff': 0.9641272649686263, 'fraction': 0.9361114173246684}
}

dist = distant_from_agg_to_orcale(optimized_voting_rules)
print('The average distance from the oracle to our optimized aggregation function is ', dist)
recalls = retrieval_vs_rule(optimized_voting_rules)
print('The recalls obtained with the optimized aggregation function are ', recalls)

aggregate_from_vote_to_SEEK(
    seek_counts_path = '/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/SEEK_dict.json',
    rules=optimized_voting_rules,
    sub_ele_pairs_path= '/archive/vision/beery/animal_reid/datasets/elephants_zooniverse/ele_id_project/data/out_apr2/sub_ele_pairs.json',
    orinal_image_dict_path  ='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_original.csv',
    export_path='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_optimized.csv',
    drop_missing_ears=True
)

The average distance from the oracle to our optimized aggregation function is  0.8883112269066938


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Concept head loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
The recalls obtained with the optimized aggregation function are  {1: 0.0456968773800457, 5: 0.07768469154607768, 10: 0.10586443259710586, 20: 0.12566641279512566, 100: 0.31607006854531605}
The recalls obtained with the optimized aggregation function are  {1: 0.0456968773800457, 5: 0.07768469154607768, 10: 0.10586443259710586, 20: 0.12566641279512566, 100: 0.31607006854531605}


(      subject_id                                              image ele_id  \
 0       92126812  EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...  T1000   
 1       92126813  EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...  T1000   
 2       92126815  EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...  T1001   
 3       92126816  EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...  T1001   
 4       92126817  EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...  T1001   
 ...          ...                                                ...    ...   
 8439    92306374  EFA_IDI/F1000/F1000_R1/EFA_IDI_F1000_R1_IMAG00...  F1000   
 8440    92306375  EFA_IDI/F1000/F1000_R1/EFA_IDI_F1000_R1_IMAG00...  F1000   
 8441    92306418  EFA_IDI/F1000/F1000_R1/EFA_IDI_F1000_R1_IMAG00...  F1000   
 8442    92306424  EFA_IDI/F1000/F1000_R1/EFA_IDI_F1000_R1_IMAG00...  F1000   
 8443    92306426  EFA_IDI/F1000/F1000_R1/EFA_IDI_F1000_R1_IMAG00...  F1000   
 
       #season  #capture                     anony

# [PART 2] Aggregation in the database

Now that we have better labels, thanks to a better aggregation during the data processing, we have to optimize the aggregation between individuals in the database.

The most realistic set up is full correction in the query and in the gallery AND an aggregation in the gallery. We will optimise the aggreegation rules to maximise the retrieval in this scenario.

In [20]:
dataset = EleHandler(subset='IDI_6', dictonary_path='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_original.csv')
train_indices, test_indices = dataset.split_perpendicular_to_elephants_and_encounters(split_sizes=[0.5,0.5], hour_delta = 0.2)

train_subset = Subset(dataset, train_indices)
test_subset = Subset(dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=True)

code ='rule_optimisation'
r = Retrieval()

MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False, pretraining='concept_w')

# Historically, aggregatate seek is a boolean to use or not the traditional heuristic rules. I will make sure that this still works.
# But I will modify the code to accept rules instesd of a boolean.
query_embeddings, query_labels = concept_head.collect_embeddings(test_loader, backbone=MD_for_concepts, intervention_fn=SEEK.perfect_correction, aggregate_seeks=False)
gallery_embeddings, gallery_labels = concept_head.collect_embeddings(train_loader, backbone=MD_for_concepts, intervention_fn=SEEK.perfect_correction, aggregate_seeks=True, aggregate_rules=traditional_rules)
test_similarity_matrix = r.similarity_matrix(query_embeddings, gallery_embeddings, distance='seek_homemade_3')

# Compute and print recalls
test_recalls = {k: r.compute_recall_at_k(similarity_matrix=test_similarity_matrix, query_labels=query_labels, gallery_labels=gallery_labels, k=k) for k in [1,5,10,20,100]}
for k in [1,5,10,20,100]: print(f"Recall@{k}: {test_recalls[k]*100:.2f}%")

/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Concept head loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
Recall@1: 4.11%
Recall@5: 8.15%
Recall@10: 10.28%
Recall@20: 13.40%
Recall@100: 32.22%
Recall@1: 4.11%
Recall@5: 8.15%
Recall@10: 10.28%
Recall@20: 13.40%
Recall@100: 32.22%


In [2]:
rules_to_test = {
'sex': {'cutoff': 1, 'fraction': 3.2846153859098934},
'age': {'cutoff': 1, 'fraction': 3.030614746927617},
'tusks': {'cutoff': 1, 'fraction': 1.3312930166044628},
'ear_most_prominent':{'cutoff': 1, 'fraction': 1.788490949619698},
'ear_least_prominent': {'cutoff': 1, 'fraction': 1.6865733430829595},
'extremes': {'cutoff': 1, 'fraction': 0.9361114173246684}
}
    
dataset = EleHandler(subset='IDI_6', dictonary_path='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_optimized.csv')
train_indices, test_indices = dataset.split_perpendicular_to_elephants_and_encounters(split_sizes=[0.5,0.5], hour_delta = 0.2)

train_subset = Subset(dataset, train_indices)
test_subset = Subset(dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=True)

code ='rule_optimisation'
r = Retrieval()

MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False, pretraining='concept_w')

query_embeddings, query_labels = concept_head.collect_embeddings(test_loader, backbone=MD_for_concepts, intervention_fn=SEEK.perfect_correction, aggregate_seeks=False)
gallery_embeddings, gallery_labels = concept_head.collect_embeddings(train_loader, backbone=MD_for_concepts, intervention_fn=SEEK.perfect_correction, aggregate_seeks=True, aggregate_rules=rules_to_test)
test_similarity_matrix = r.similarity_matrix(query_embeddings, gallery_embeddings, distance='seek_homemade_3')

# Compute and print recalls
test_recalls = {k: r.compute_recall_at_k(similarity_matrix=test_similarity_matrix, query_labels=query_labels, gallery_labels=gallery_labels, k=k) for k in [1,5,10,20,100]}
for k in [1,5,10,20,100]: print(f"Recall@{k}: {test_recalls[k]*100:.2f}%")

NameError: name 'EleHandler' is not defined

In [ ]:
import optuna, torch
from optuna.samplers import TPESampler
from torch.utils.data import DataLoader, Subset

from seek_code import SEEK
from backbone import Backbone
from retrieval import Retrieval
from projector import Projector
from data_handler import EleHandler
from concept_head_tunneled import ConceptHeadTunneled, categorical_CE_loss, CrossedHeadNN, MultiHeadNN

from pathlib import Path
from torch.utils.data import DataLoader, Subset
import os,json
import pandas as pd

# --- Start near your traditional rules ---------------------------------------
traditional_rules = {
    'sex': {'cutoff': 0.8, 'fraction': 2.0},
    'age': {'cutoff': 0.8, 'fraction': 1.0},
    'tusks': {'cutoff': 0.8, 'fraction': 1.0},
    'ear_most_prominent': {'cutoff': 0.9, 'fraction': 2.0},
    'ear_least_prominent': {'cutoff': 0.95, 'fraction': 2.5},
    'extremes': {'cutoff': 0.8, 'fraction': 1.5},
}

# Wide, permissive bounds for the optimizer
_BOUNDS = {
    k: {
        "cutoff": (0.2, 1.0),
        "fraction": (0.5, 3.5)  # search on log-scale
    } for k in traditional_rules
}

# --- One-time setup (queries fixed) ------------------------------------------
def setup(dictonary_path):
    dataset = EleHandler(subset='IDI_6', dictonary_path=dictonary_path)
    tr_idx, te_idx = dataset.split_perpendicular_to_elephants_and_encounters([0.5, 0.5], hour_delta=0.2)
    train_loader = DataLoader(Subset(dataset, tr_idx), batch_size=64, shuffle=True, num_workers=4)
    test_loader  = DataLoader(Subset(dataset, te_idx),  batch_size=64, shuffle=False, num_workers=4)

    code = 'rule_opt_minimal'
    r = Retrieval()
    MD = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
    head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code=code,
                               layer=CrossedHeadNN(), reset_weights=False, pretraining='concept_w')

    with torch.no_grad():
        q_emb, q_lab = head.collect_embeddings(test_loader, backbone=MD,
                                               intervention_fn=SEEK.perfect_correction,
                                               aggregate_seeks=False)
    return dict(dataset=dataset, train_loader=train_loader, retrieval=r, MD=MD, head=head,
                q_emb=q_emb, q_lab=q_lab)

PIPE = setup("/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_optimized.csv")

# --- Minimal objective: maximize Recall@1 ------------------------------------
def _suggest_rules(trial):
    rules = {}
    for f, b in _BOUNDS.items():
        rules[f] = {
            "cutoff":   trial.suggest_float(f"{f}.cutoff", *b["cutoff"]),
            "fraction": trial.suggest_float(f"{f}.fraction", *b["fraction"], log=True),
        }
    return rules

def objective(trial):
    rules = _suggest_rules(trial)
    with torch.no_grad():
        gal_emb, gal_lab = PIPE["head"].collect_embeddings(
            PIPE["train_loader"], backbone=PIPE["MD"],
            intervention_fn=SEEK.perfect_correction,
            aggregate_seeks=True, aggregate_rules=rules
        )
        sim = PIPE["retrieval"].similarity_matrix(PIPE["q_emb"], gal_emb, distance='seek_homemade_3')
        r1 = PIPE["retrieval"].compute_recall_at_k(similarity_matrix=sim,
                                                   query_labels=PIPE["q_lab"],
                                                   gallery_labels=gal_lab, k=1)
    return float(r1) if r1 is not None else 0.0

def tune_rules_for_recall1(n_trials=100, seed=0):
    study = optuna.create_study(direction="maximize",
                                sampler=TPESampler(seed=seed, n_startup_trials=min(20, n_trials//5)))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = {f: {"cutoff": study.best_params[f"{f}.cutoff"],
                "fraction": study.best_params[f"{f}.fraction"]} for f in _BOUNDS}
    return best, study.best_value

# --- Run ----------------------------------------------------------------------
best_rules, best_r1 = tune_rules_for_recall1(n_trials=5, seed=42)
print("Best Recall@1:", best_r1)
print("Best rules:")
for k, v in best_rules.items(): print(k, v)


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Concept head loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt


/data/vision/beery/scratch/antoine/CBM_reid/methods/concept_head_tunneled.py:423: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(path, map_loc

Best Recall@1: 0.02513328255902513
Best rules:
sex {'cutoff': 0.5662570016976377, 'fraction': 0.505869169523547}
age {'cutoff': 0.8396121299403545, 'fraction': 0.5445675510677364}
tusks {'cutoff': 0.46529401155806127, 'fraction': 2.720519028260534}
ear_most_prominent {'cutoff': 0.2219103111792452, 'fraction': 1.0730777111479513}
ear_least_prominent {'cutoff': 0.9195853915028844, 'fraction': 0.5121228451660323}
extremes {'cutoff': 0.742603039801002, 'fraction': 1.4576109139131155}


### Results, after running this for 2500 iterations, I obtain the following results

Best Recall@1: 0.05483625285605483


In [3]:

optimized_agg_rules_database = {
'sex': {'cutoff': 0.9334337305774126, 'fraction': 0.7151849388533961},
'age': {'cutoff': 0.9514760451994138, 'fraction': 0.5510405823660589},
'tusks': {'cutoff': 0.4369025581005079, 'fraction': 1.8715870244928936},
'ear_most_prominent':{'cutoff': 0.8346892883784236, 'fraction': 2.432201122842605},
'ear_least_prominent': {'cutoff': 0.3902627093863492, 'fraction': 2.0680981200569604},
'extremes': {'cutoff': 0.46972912427893604, 'fraction': 0.8622955318459603}
}

dataset = EleHandler(subset='IDI_6', dictonary_path='/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary_optimized.csv')
train_indices, test_indices = dataset.split_perpendicular_to_elephants_and_encounters(split_sizes=[0.5,0.5], hour_delta = 0.2)

train_subset = Subset(dataset, train_indices)
test_subset = Subset(dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=True)

code ='rule_optimisation'
r = Retrieval()

MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False, pretraining='concept_w')

query_embeddings, query_labels = concept_head.collect_embeddings(test_loader, backbone=MD_for_concepts, intervention_fn=SEEK.perfect_correction, aggregate_seeks=False)
gallery_embeddings, gallery_labels = concept_head.collect_embeddings(train_loader, backbone=MD_for_concepts, intervention_fn=SEEK.perfect_correction, aggregate_seeks=True, aggregate_rules=optimized_agg_rules_database)
test_similarity_matrix = r.similarity_matrix(query_embeddings, gallery_embeddings, distance='seek_homemade_3')

# Compute and print recalls
test_recalls = {k: r.compute_recall_at_k(similarity_matrix=test_similarity_matrix, query_labels=query_labels, gallery_labels=gallery_labels, k=k) for k in [1,5,10,20,100]}
for k in [1,5,10,20,100]: print(f"Recall@{k}: {test_recalls[k]*100:.2f}%")

/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Concept head loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
Recall@1: 5.33%
Recall@5: 7.54%
Recall@10: 9.29%
Recall@20: 13.40%
Recall@100: 26.73%
